# sparse_knn on a commercial model — gpt-5.6-sol, no thinking

---
## 1 — Host and working tree

Selection runs locally on the embedding index, so a GPU runtime only shortens the index
build. Generation is API-only.

In [2]:
%cd /home/prnamhr/projects/Style-Aware-MT

/home/prnamhr/projects/Style-Aware-MT


In [3]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# The pipeline is text-only and these three ship against a torch the pins contradict.
%pip uninstall -q -y torchvision torchaudio torchcodec

Note: you may need to restart the kernel to use updated packages.


In [5]:
import getpass
import logging
import os

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
logging.getLogger('httpx').setLevel(logging.WARNING)
print('OPENAI_API_KEY set')

OPENAI_API_KEY set


---
## 2 — Run parameters and the matched-contrast gate

In [19]:
import hashlib
import json
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import yaml

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'val'
ARM, ARM_NAME = 'sparse_knn', 'gpt56_sparse_knn'
# The Qwen run of the same condition is the comparator: one selection method, two
# generators. It is already scored on every metric, so the contrast costs nothing.
REF = 'sparse_knn'
CONFIG = Path('configs/commercial_gpt56_sparse_knn.yaml')
QWEN_CONFIG, BASE_CONFIG = Path('configs/sparse_knn.yaml'), Path('configs/base_qwen.yaml')
OUT = Path('outputs')
DIAG = Path(f'results/sparse_selection_{SPLIT}.json')

N_BOOT, N_STYLO, ALPHA, SEED = 10000, 2000, 0.05, 42

CFG = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
GEN, RETR, SPA = CFG['generator'], CFG['retrieval'], CFG['sparse']
RAR, PROMPT = CFG['rarity'], CFG['prompt']
RARITY_PATH = Path(RAR['out'])
print(f"{GEN['model']}: {ARM} against {REF} on Qwen, k={RETR['k']}, up to m={SPA['m']} rare, "
      f"{RAR['freeze_n']} rarest terms at df >= {RAR['min_df']}")

gpt-5.6-sol: sparse_knn against sparse_knn on Qwen, k=8, up to m=4 rare, 500 rarest terms at df >= 40


In [20]:
QWEN = yaml.safe_load(QWEN_CONFIG.read_text(encoding='utf-8'))
for block in ('prompt', 'retrieval', 'rarity', 'sparse', 'data'):
    assert CFG[block] == QWEN[block], f'{block} differs from {QWEN_CONFIG}: {CFG[block]}'
assert set(CFG) == set(QWEN), sorted(set(CFG) ^ set(QWEN))
assert RETR == yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))['retrieval'], RETR
print(f'{CONFIG.name} differs from {QWEN_CONFIG.name} in the generator and output name only')

commercial_gpt56_sparse_knn.yaml differs from sparse_knn.yaml in the generator and output name only


In [21]:
# A thinking budget would let deliberation, not selection, carry the contrast.
assert GEN['reasoning_effort'] == 'none', GEN['reasoning_effort']
assert GEN['temperature'] is None, 'gpt-5.x rejects a custom temperature; leave it null'
assert GEN['seed'] == 42 and GEN['pricing'], GEN
assert CFG['data']['eval_file'].endswith(f'{SPLIT}.jsonl'), 'test split stays sealed'
assert CFG['data']['limit'] is None, 'limit must be null for the full pass'
assert ARM_NAME != ARM, 'the output name collides with the Qwen run of the same condition'
print(f"{GEN['model']} at reasoning_effort={GEN['reasoning_effort']}, "
      f"${GEN['pricing'][0]:.2f}/${GEN['pricing'][1]:.2f} per 1M in/out")

gpt-5.6-sol at reasoning_effort=none, $4.00/$20.00 per 1M in/out


In [22]:
VAL = [json.loads(x) for x in Path(CFG['data']['eval_file']).open(encoding='utf-8') if x.strip()]
SRC = [r['input'] for r in VAL]

QWEN_ROWS = [json.loads(x) for x in (OUT / f'{REF}_{SPLIT}.jsonl').open(encoding='utf-8')
             if x.strip()]
assert [r['input'] for r in QWEN_ROWS] == SRC, f'{REF} is not aligned to {SPLIT}.jsonl'
print(f'{len(VAL)} {SPLIT} segments; comparator {REF} on {QWEN_ROWS[0]["model"]}')

1323 val segments; comparator sparse_knn on Qwen/Qwen2.5-7B-Instruct


---
## 3 — The rarity list and the index

In [23]:
RARITY = json.loads(RARITY_PATH.read_text(encoding='utf-8'))
RARITY_SHA = hashlib.sha256(RARITY_PATH.read_bytes()).hexdigest()

for key in ('min_df', 'freeze_n', 'zwnj'):
    assert RARITY['config'][key] == RAR[key], (key, RARITY['config'][key], RAR[key])
assert len(RARITY['terms']) == RAR['freeze_n'], f"{len(RARITY['terms'])} terms on the list"
assert RAR['min_df'] <= RARITY['df_observed'][0], RARITY['df_observed']
print(f"{RARITY['n_frozen']} frozen terms of {RARITY['n_eligible']} eligible, "
      f"realized df {RARITY['df_observed']}, sha256 {RARITY_SHA[:16]}...")

500 frozen terms of 525 eligible, realized df [40, 410], sha256 8fa5b0b2c0dbf344...


In [24]:
# data/knn_index is git-ignored, so a fresh session rebuilds it. It must be the index the
# Qwen arms retrieved from, hence base_qwen.yaml rather than this config.
INDEX = Path(RETR['index_dir'])
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
if not all((INDEX / f).exists() for f in INDEX_FILES):
    !{PY} manage.py build_index --config configs/base_qwen.yaml

meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == RETR['embed_model'], meta
print(f"{meta['n_passages']} pool passages on {meta['embed_model']}, dim {meta['dim']}")

10860 pool passages on intfloat/multilingual-e5-large-instruct, dim 1024


---
## 4 — Routing

Selection reads the query and the index, not the generator, so the routes here must be the
ones the committed diagnostic recorded for the Qwen arm.

In [25]:
from src.retrieval.rarity import load_irregular
from src.retrieval.retrieve import RetrievalIndex
from src.retrieval.sparse import ROUTES, SparseRetriever

index = RetrievalIndex(RETR['index_dir'], embed_model=RETR['embed_model'])
retriever = SparseRetriever(
    index, load_irregular(str(RARITY_PATH)), index, zwnj=RAR['zwnj'], m=SPA['m'],
)
SELECTED, TRACES = retriever.select_with_trace(SRC, k=RETR['k'])
print(f'{len(TRACES)} traces, {len(SELECTED[0])} exemplars per prompt')

Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3031.25it/s]


1323 traces, 8 exemplars per prompt


In [26]:
SLOTS = min(SPA['m'], RETR['k'])
HIST = {str(v): sum(t['n_sparse'] == v for t in TRACES) for v in range(SLOTS + 1)}
ROUTE_COUNTS = {r: sum(t['route'] == r for t in TRACES) for r in ROUTES}

diag = json.loads(DIAG.read_text(encoding='utf-8'))
assert diag['config']['index_dir'] == RETR['index_dir'], diag['config']
assert HIST == diag['n_sparse']['histogram'], f'{HIST} against {diag["n_sparse"]["histogram"]}'
assert ROUTE_COUNTS == diag['routes'], f'{ROUTE_COUNTS} against {diag["routes"]}'

ROUTED = len(TRACES) - ROUTE_COUNTS['dense']
print(f'routes {ROUTE_COUNTS}  ({ROUTED / len(TRACES):.1%} routed), matches {DIAG}')
print(f'rare slots filled {HIST}, mean {np.mean([t["n_sparse"] for t in TRACES]):.3f}')

routes {'full': 563, 'partial': 639, 'dense': 121}  (90.9% routed), matches results/sparse_selection_val.json
rare slots filled {'0': 121, '1': 208, '2': 252, '3': 179, '4': 563}, mean 2.646


---
## 5 — The prompts the two arms will be sent

In [27]:
from src.infer.run import _load_configured_glossary, build_fewshot_user, order_exemplars

K = RETR['k']
STYLE = Path(PROMPT['style_instruction_file']).read_text(encoding='utf-8')
GLOSSARY = _load_configured_glossary(CFG)
ORDERED = [order_exemplars(ex, PROMPT['ordering']) for ex in SELECTED]
PROMPTS = [build_fewshot_user(s, ex, GLOSSARY) for s, ex in zip(SRC, ORDERED)]

# The dense-only prompts are never sent. They are built so section 9 can check the rarity
# channel at the prompt rather than by buying a second arm to diff predictions against.
DENSE_ORDERED = [order_exemplars(ex, PROMPT['ordering']) for ex in index.retrieve(SRC, k=K)]
DENSE_PROMPTS = [build_fewshot_user(s, ex, GLOSSARY) for s, ex in zip(SRC, DENSE_ORDERED)]

for i, ex in enumerate(ORDERED):
    keys = [(e['input'], e['output']) for e in ex]
    assert len(keys) == K, f'segment {i}: {len(keys)} exemplars, expected {K}'
    assert len(set(keys)) == len(keys), f'segment {i}: an exemplar repeats across the channels'
    assert SRC[i] not in {e['input'] for e in ex}, f'segment {i}: the query is its own exemplar'

CHARS = np.array([len(p) for p in PROMPTS])
print(f'{ARM_NAME:20s} median {np.median(CHARS):6.0f}  mean {CHARS.mean():6.0f}  '
      f'max {CHARS.max():6d} chars')

gpt56_sparse_knn     median   5418  mean   5429  max   9895 chars


In [28]:
i = int(np.flatnonzero(np.array([t['n_sparse'] for t in TRACES]) == SLOTS)[0])
print(STYLE)
print('=' * 88)
print(PROMPTS[i])

You are an expert translator of Bahá'í scripture from Persian and Arabic into English.

Render the source text into English in the formal, elevated, scriptural register of Shoghi Effendi's authorized translations. Observe the following:

- Preserve the dignity and cadence of sacred prose; favour the elevated, archaic register over modern neutral English.
- Use the second-person sacred pronouns and their verb forms where the source addresses the Divine or is addressed by It: "Thou", "Thee", "Thy", "Thine", and verb endings such as "art", "hast", "dost", "doth".
- Retain formal vocatives such as "O" and honorific constructions where the source warrants them.
- Translate the full meaning faithfully; do not add commentary, explanation, transliteration, or footnotes.
- Output only the English translation, as a single continuous passage with no quotation marks, labels, or preamble.

Here are example translations in the required style:

[Terms] الله → God | قل → Say
Source: لو تسمع صریر القلم

---
## 6 — Pilot, so the full pass is priced by measurement

In [29]:
N_PILOT = 15
PILOT_CAP_USD = 1.00

# A working ceiling on Persian tokenisation; the pilot replaces it with the model's counts.
TOK_PER_CHAR = 0.8
in_rate, out_rate = GEN['pricing']
guess = N_PILOT * (
    TOK_PER_CHAR * (CHARS.mean() + len(STYLE)) * in_rate + 256 * out_rate) / 1e6
assert guess <= PILOT_CAP_USD, f'pilot projects to ${guess:.2f} over the ${PILOT_CAP_USD:.2f} cap'
print(f'{N_PILOT} pilot calls, ${guess:.2f} at the ceiling rate')

15 pilot calls, $0.38 at the ceiling rate


In [33]:
PILOT_CONFIG = Path(tempfile.gettempdir()) / 'gpt56_sparse_knn_pilot.yaml'
_pilot = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
_pilot['data']['limit'] = N_PILOT
PILOT_CONFIG.write_text(yaml.safe_dump(_pilot, sort_keys=False), encoding='utf-8')

USAGE_PATH = OUT / f'{ARM_NAME}_{SPLIT}_usage.json'


def priced_usage(snapshot: Path) -> dict:
    """Return the usage of the pass that was paid for, not of a no-op resume."""
    # infer resumes, so re-running a finished pass makes no calls and rewrites the usage
    # file with zeros. Snapshot the priced copy and read that back instead.
    u = json.loads(USAGE_PATH.read_text(encoding='utf-8'))
    if u['calls']:
        snapshot.write_text(json.dumps(u, indent=2), encoding='utf-8')
    assert snapshot.exists(), (
        f'this pass made no calls and no priced snapshot exists at {snapshot}; the '
        f'output file is already complete and its cost was overwritten')
    return json.loads(snapshot.read_text(encoding='utf-8'))


r = subprocess.run([PY, 'manage.py', 'infer', '--condition', ARM,
                    '--config', str(PILOT_CONFIG), '--out-name', ARM_NAME], check=False)
assert r.returncode == 0, f'{ARM_NAME} pilot exited {r.returncode}'
PILOT_USAGE = priced_usage(OUT / f'{ARM_NAME}_{SPLIT}_pilot_usage.json')

sparse_knn: k=8 as up to 4 rarity + cosine for 15 ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 888.81it/s]


  routes: {'full': 8, 'partial': 5, 'dense': 2}, mean rare slots filled: 3.00
Output name overridden: condition 'sparse_knn' -> outputs/gpt56_sparse_knn_val.jsonl
resuming gpt56_sparse_knn: 15/15 already done
Generating 15 translations with gpt-5.6-sol (sparse_knn) ...
Wrote outputs/gpt56_sparse_knn_val.jsonl
Usage: {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'cost_usd': 0.0}


In [34]:
assert PILOT_USAGE['calls'] == N_PILOT, f"{PILOT_USAGE['calls']} pilot calls, expected {N_PILOT}"
PILOT_SPENT = PILOT_USAGE['cost_usd']
RATE = PILOT_SPENT / PILOT_USAGE['calls']
PROJECTED = RATE * (len(VAL) - N_PILOT)
print(f"{ARM_NAME:20s} {PILOT_USAGE['prompt_tokens'] / N_PILOT:7.0f} in / "
      f"{PILOT_USAGE['completion_tokens'] / N_PILOT:5.0f} out per call   ${RATE:.4f} each")
print(f'pilot spent ${PILOT_SPENT:.2f}; the remaining {len(VAL) - N_PILOT} calls '
      f'project to ${PROJECTED:.2f}')

gpt56_sparse_knn        1925 in /    45 out per call   $0.0086 each
pilot spent $0.13; the remaining 1308 calls project to $11.25


In [35]:
# Left False so a top-to-bottom re-run cannot authorise itself.
SPEND_OK = True
BUDGET_USD = 15.00
assert PROJECTED <= BUDGET_USD, f'projection ${PROJECTED:.2f} exceeds ${BUDGET_USD:.2f}'
print(f'authorised {SPEND_OK}   cap ${BUDGET_USD:.2f}   projected ${PROJECTED:.2f}')

authorised True   cap $15.00   projected $11.25


---
## 7 — Generation

In [36]:
assert SPEND_OK, 'set SPEND_OK = True in the cell above to authorise the full pass'
t0 = time.perf_counter()
r = subprocess.run([PY, 'manage.py', 'infer', '--condition', ARM, '--config', str(CONFIG),
                    '--out-name', ARM_NAME], check=False)
assert r.returncode == 0, f'{ARM_NAME} exited {r.returncode}'
GEN_USAGE = priced_usage(OUT / f'{ARM_NAME}_{SPLIT}_full_usage.json')
GEN_SECONDS = round(time.perf_counter() - t0, 1)
print(f'{GEN_SECONDS / 60:.1f} min, finished {datetime.now(timezone.utc).isoformat()}')

sparse_knn: k=8 as up to 4 rarity + cosine for 1323 ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3585.47it/s]


  routes: {'full': 563, 'partial': 639, 'dense': 121}, mean rare slots filled: 2.65
Output name overridden: condition 'sparse_knn' -> outputs/gpt56_sparse_knn_val.jsonl
resuming gpt56_sparse_knn: 15/1323 already done
Generating 1323 translations with gpt-5.6-sol (sparse_knn) ...
  20/1323
  25/1323
  30/1323
  35/1323
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323

---
## 8 — The outputs, their provenance, and the bill

In [37]:
ROWS = [json.loads(x) for x in (OUT / f'{ARM_NAME}_{SPLIT}.jsonl').open(encoding='utf-8')
        if x.strip()]
assert len(ROWS) == len(VAL), f'{ARM_NAME}: {len(ROWS)} rows, expected {len(VAL)}'
assert [r['input'] for r in ROWS] == SRC, f'{ARM_NAME}: source order differs from the eval file'
assert all(r['model'] == GEN['model'] for r in ROWS), f'{ARM_NAME}: a different model'
errored = [i for i, r in enumerate(ROWS) if 'error' in r]
blank = [i for i, r in enumerate(ROWS) if not r['prediction'].strip()]
assert not errored, f'{ARM_NAME}: {len(errored)} segments recorded an error: {errored[:5]}'
print(f'{ARM_NAME:20s} {len(ROWS)} rows, {len(blank)} blank, {len(errored)} errors')

gpt56_sparse_knn     1323 rows, 0 blank, 0 errors


In [38]:
PROV = GEN_USAGE['provenance']
assert PROV['rarity_sha256'] == RARITY_SHA, f"{PROV['rarity_sha256']} != {RARITY_SHA}"
assert PROV['k'] == RETR['k'] and PROV['m'] == SPA['m'], PROV
assert all(PROV[key] == RAR[key] for key in ('min_df', 'freeze_n')), PROV
assert PROV['ordering'] == PROMPT['ordering'], PROV
print(json.dumps(PROV, indent=2))

{
  "k": 8,
  "ordering": "most_similar_last",
  "index_dir": "data/knn_index",
  "m": 4,
  "min_df": 40,
  "freeze_n": 500,
  "rarity_file": "results/rarity_train.json",
  "rarity_sha256": "8fa5b0b2c0dbf3441254c1ce6ad6a0a91a50fd90583a18f1991f5fa263ca8d49"
}


In [39]:
# GEN_USAGE covers the resumed calls only; the pilot rows were paid for separately.
SPENT = PILOT_USAGE['cost_usd'] + GEN_USAGE['cost_usd']
CALLS = PILOT_USAGE['calls'] + GEN_USAGE['calls']
assert SPENT <= BUDGET_USD + PILOT_CAP_USD, f'${SPENT:.2f} spent against the caps'
print(f"{GEN_USAGE['calls']} resumed calls  ${GEN_USAGE['cost_usd']:.2f}")
print(f'{CALLS} paid generation calls, ${SPENT:.2f} on {GEN["model"]} in total')

1308 resumed calls  $10.16
1323 paid generation calls, $10.29 on gpt-5.6-sol in total


---
## 9 — The rarity channel, checked at the prompt

In [42]:
DIFFERS = np.array([a != b for a, b in zip(PROMPTS, DENSE_PROMPTS)])
ROUTE = np.array([t['route'] for t in TRACES])
NS = np.array([t['n_sparse'] for t in TRACES])

print(f'{DIFFERS.sum()}/{len(DIFFERS)} prompts differ from dense-only ({DIFFERS.mean():.1%}); '
      f'routed fraction is {ROUTED / len(TRACES):.1%}')
for r in ROUTES:
    sel = ROUTE == r
    print(f'  {r:8s} n={sel.sum():5d}  differ {DIFFERS[sel].mean():6.1%}')
print('by dose:')
for v in range(SLOTS + 1):
    sel = NS == v
    if sel.any():
        print(f'  n_sparse={v}  n={sel.sum():5d}  differ {DIFFERS[sel].mean():6.1%}')

928/1323 prompts differ from dense-only (70.1%); routed fraction is 90.9%
  full     n=  563  differ  94.1%
  partial  n=  639  differ  62.3%
  dense    n=  121  differ   0.0%
by dose:
  n_sparse=0  n=  121  differ   0.0%
  n_sparse=1  n=  208  differ  38.0%
  n_sparse=2  n=  252  differ  71.4%
  n_sparse=3  n=  179  differ  77.7%
  n_sparse=4  n=  563  differ  94.1%


In [43]:
# The failure mode is a rarity pick that never reaches the prompt; overlap with the dense
# top-k is expected and is not one, so the invariant is asserted and divergence reported.
missing = sum(
    (index.pairs[int(row)]['input'], index.pairs[int(row)]['output'])
    not in {(e['input'], e['output']) for e in sel}
    for t, sel in zip(TRACES, SELECTED) for row in t['sparse_rows'])
assert missing == 0, f'{missing} served rarity exemplars never reached the exemplar set'
assert not DIFFERS[ROUTE == 'dense'].any(), (
    'a dense-routed segment built a prompt the dense selector would not have built')

routed_share, full_share = DIFFERS[ROUTE != 'dense'].mean(), DIFFERS[NS == SLOTS].mean()
assert full_share >= 0.90, f'full-dose segments diverge on only {full_share:.1%}'
assert routed_share >= 0.70, f'routed segments diverge on only {routed_share:.1%}'
assert (np.diff([DIFFERS[NS == v].mean() for v in range(SLOTS + 1)]) > 0).all(), (
    'divergence is not monotone in dose')
print(f'every rarity pick reaches the prompt; dense-routed prompts are identical; '
      f'{routed_share:.1%} of routed and {full_share:.1%} of full-dose segments differ')

every rarity pick reaches the prompt; dense-routed prompts are identical; 77.2% of routed and 94.1% of full-dose segments differ


In [44]:
i = int(np.flatnonzero((ROUTE == 'full') & DIFFERS)[0])
print('SOURCE  :', SRC[i][:110])
print('terms   :', TRACES[i]['query_terms'][:8], f"served {TRACES[i]['served_terms']}")
print('REFERENCE:', VAL[i]['output'][:200])
print(f'{ARM_NAME:20s}:', ROWS[i]['prediction'][:200])
print(f'{REF + " (qwen)":20s}:', QWEN_ROWS[i]['prediction'][:200])

SOURCE  : جواهر الأسرار فی معارج الأسفار لمن اراد ان یتقرّب بالله المقتدر الغفّار فهنیاً للأبرار الّذین یشربون من هذه ال
terms   : ['اراد', 'بالله', 'هذه', 'لمن', 'المقتدر', 'الذین'] served ['اراد', 'بالله', 'هذه', 'لمن']
REFERENCE: The essence of the divine mysteries in the journeys of ascent set forth for those who long to draw nigh unto God, the Almighty, the Ever-Forgiving—blessed be the righteous that quaff from these crysta
gpt56_sparse_knn    : The Gems of Divine Mysteries concerning the stages of the journey for him who desireth to draw nigh unto God, the Almighty, the Ever-Forgiving. Blessed, then, are the righteous who drink from these ri
sparse_knn (qwen)   : O God, the pearls of mysteries lie hidden in the stations of journeys for him who desireth to draw nigh unto Thee, the All-Powerful, the Forgiver. Verily, a blessed lot for the righteous who drink fro


---
## 10 — Free metrics

In [45]:
CONDS = [ARM_NAME]
READOUT = [ARM_NAME, REF]
!{PY} manage.py eval --conditions {' '.join(READOUT)} --split {SPLIT}

condition         n     BLEU   chrF   marker_rate  ref_marker_rate
------------------------------------------------------------------
gpt56_sparse_knn  1323  26.5   51.48  1.11         0.93           
sparse_knn        1323  14.31  40.08  1.25         0.93           


In [46]:
from src.eval.quick import score

SURFACE = {c: score(c, OUT, SPLIT) for c in READOUT}
print(f"{'condition':22s} {'chrF':>8s} {'BLEU':>8s} {'markers/seg':>12s}")
for cond in READOUT:
    s = SURFACE[cond]
    print(f"{cond:22s} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")
print(f"gold targets carry {SURFACE[REF]['ref_marker_rate']:.2f} markers per segment")

condition                  chrF     BLEU  markers/seg
gpt56_sparse_knn          51.48    26.50         1.11
sparse_knn                40.08    14.31         1.25
gold targets carry 0.93 markers per segment


In [47]:
!{PY} manage.py stylometrics --conditions {' '.join(READOUT)} --split {SPLIT} --targets-split train

label             n      lex_density  lex_density_sd  ttr     ttr_sd  root_ttr  root_ttr_sd  sent_len_mean  sent_len_mean_sd  sent_len_var  sent_len_var_sd  marker_rate  marker_rate_sd  stylo_dist
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
target:train      10860  0.4344       0.1101          0.854   0.1085  4.0437    1.0426       24.7388        16.6125           6.6894        58.4011          0.0573       0.0782          0.0       
gpt56_sparse_knn  1323   0.4119       0.1034          0.8496  0.1153  3.9506    1.0032       24.5305        16.6304           4.8057        41.4846          0.05         0.0674          0.2455    
sparse_knn        1323   0.4027       0.102           0.8357  0.1199  3.8645    0.9432       24.1243        15.8286           4.8616        42.7644          0.0571       0.0794          0.375     


In [48]:
STYLO_PATH = f'results/stylometrics_ci_{ARM_NAME}_{SPLIT}.json'
!{PY} manage.py stylometrics_ci --split {SPLIT} --conditions {' '.join(READOUT)} \
    --n_resamples {N_STYLO} --alpha {ALPHA} --seed {SEED} --results_path {STYLO_PATH}


Register fit of the main conditions  (split=val, n=1323 segments, resamples=2000, seed=42)
stylo_dist = standardized distance to the target-register centroid; lower is better.

rank  condition         stylo_dist  ci95              P(this rank)  modal rank  mean rank
-----------------------------------------------------------------------------------------
1     gpt56_sparse_knn  0.2455      [0.2029, 0.2957]  1.000         1 (1.000)   1.00     
2     sparse_knn        0.3750      [0.3291, 0.4263]  1.000         2 (1.000)   2.00     

Signed z per register feature (95% CI; 0 = on target)
condition         lex_density              ttr                      root_ttr                 marker_rate            
--------------------------------------------------------------------------------------------------------------------
gpt56_sparse_knn  -0.205 [-0.256, -0.154]  -0.040 [-0.097, 0.015]   -0.089 [-0.141, -0.037]  -0.094 [-0.139, -0.047]
sparse_knn        -0.288 [-0.339, -0.237]  -0.168 [-0.22

In [49]:
STYLO = json.loads(Path(STYLO_PATH).read_text(encoding='utf-8'))
for cond in READOUT:
    print(f"  {cond:22s} stylo_dist {STYLO['cells'][cond]['stylo_dist']:.4f}")

  gpt56_sparse_knn       stylo_dist 0.2455
  sparse_knn             stylo_dist 0.3750


### COMET

In [50]:
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)

COMET_PATH = f'results/comet_{SPLIT}.json'
PRIOR_COMET = set(json.loads(Path(COMET_PATH).read_text(encoding='utf-8')))
r = subprocess.run([COMET_PY, 'manage.py', 'comet', '--conditions', *CONDS, '--split', SPLIT,
                    '--results_path', COMET_PATH, '--batch_size', '16'], check=False)
assert r.returncode == 0, f'comet exited {r.returncode}'

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2598.70it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and ver

gpt56_sparse_knn COMET 0.7480  (n=1323)
preserved 15 condition(s) not scored here: afsp_full, afsp_full_casefix, afsp_margin, commercial_haiku, knn_fewshot, peft, peft_afsp, peft_afsp_casefix, peft_knn, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, sparse_knn, zeroshot
Wrote results/comet_val.json


In [51]:
COMET = json.loads(Path(COMET_PATH).read_text(encoding='utf-8'))
assert PRIOR_COMET <= set(COMET), f'lost from {COMET_PATH}: {sorted(PRIOR_COMET - set(COMET))}'
for cond in READOUT:
    assert COMET[cond]['sources'] == COMET[REF]['sources'], f'{cond} is not paired'
    print(f"  {cond:22s} COMET {COMET[cond]['system']:.4f}")

  gpt56_sparse_knn       COMET 0.7480
  sparse_knn             COMET 0.6846


---
## 11 — Judge Φ (paid)

In [52]:
JUDGE_CFG = 'configs/judge_eval.yaml'
JUDGE_RESULTS = f'results/judge_{SPLIT}.json'
JUDGE_USAGE = f'results/judge_{SPLIT}_usage.json'
JUDGE_CI_PATH = f'results/judge_ci_{ARM_NAME}_{SPLIT}.json'

PRIOR_JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
PRIOR_USAGE = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
PRIOR_SPEND, PRIOR_CALLS = (PRIOR_USAGE['cumulative'][k] for k in ('cost_usd', 'calls'))

assert REF in PRIOR_JUDGE, f'{REF} carries no Phi; the comparator would have to be bought too'
BUY = [c for c in CONDS if c not in PRIOR_JUDGE]
PER_CALL = PRIOR_SPEND / PRIOR_CALLS
N_CALLS = len(VAL) * len(BUY)
PROJECTED_J = PER_CALL * N_CALLS
print(f"buying Phi for {BUY or 'nothing'}: {N_CALLS} calls at ${PER_CALL:.5f} "
      f"= ${PROJECTED_J:.2f} projected (cumulative judge spend ${PRIOR_SPEND:.2f})")

buying Phi for ['gpt56_sparse_knn']: 1323 calls at $0.00102 = $1.35 projected (cumulative judge spend $8.09)


In [54]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

# Left False so a top-to-bottom re-run cannot authorise itself.
SPEND_OK_J =    True
BUDGET_J_USD = 2.00
assert PROJECTED_J <= BUDGET_J_USD, f'projection ${PROJECTED_J:.2f} over ${BUDGET_J_USD:.2f}'
print(f'authorised {SPEND_OK_J}   cap ${BUDGET_J_USD:.2f}   projected ${PROJECTED_J:.2f}')

authorised True   cap $2.00   projected $1.35


In [55]:
# Re-running over a complete cache makes no request.
if not BUY:
    print('nothing to buy: the arm already carries Phi')
else:
    assert SPEND_OK_J, 'set SPEND_OK_J = True in the cell above to authorise the pass'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG], check=False)
    assert r.returncode == 0, f'judge exited {r.returncode}'

judge claude-haiku-4-5  tag=(none)  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 1323 segments for gpt56_sparse_knn with claude-haiku-4-5 ...
  gpt56_sparse_knn Φ 3.625  (coverage 100%)
preserved 13 condition(s) not re-scored: afsp_full, afsp_margin, commercial_haiku, knn_fewshot, peft, peft_afsp, peft_knn, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, sparse_knn, zeroshot
Wrote results/judge_val.json
Judge usage: {'calls': 1323, 'prompt_tokens': 627878, 'completion_tokens': 141730, 'cost_usd': 1.3365}
Wrote results/judge_val_usage.json  (cumulative $9.42)


In [56]:
JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
HAVE_PHI = all(c in JUDGE for c in READOUT)
lost = sorted(set(PRIOR_JUDGE) - set(JUDGE))
assert not lost, f'lost from {JUDGE_RESULTS}: {lost}'

usage = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
assert usage['priced'], 'the judge model has no pricing table; cost_usd is a floor, not a bill'
SPENT_J = usage['cumulative']['cost_usd'] - PRIOR_SPEND
assert SPENT_J <= BUDGET_J_USD, f'${SPENT_J:.2f} spent against a ${BUDGET_J_USD:.2f} cap'

if HAVE_PHI:
    for cond in READOUT:
        assert JUDGE[cond]['model'] == JUDGE[REF]['model'], f'{cond}: two raters, not one'
    r = subprocess.run([PY, 'manage.py', 'judge_ci', '--split', SPLIT, '--conditions', *READOUT,
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--results_path', JUDGE_CI_PATH], check=False)
    assert r.returncode == 0, f'judge_ci exited {r.returncode}'

METRICS = ['chrf', 'bleu', 'comet'] + (['judge'] if HAVE_PHI else [])
print(f"{usage['cumulative']['calls'] - PRIOR_CALLS} paid calls, ${SPENT_J:.2f} on "
      f"{usage['model']}; reading out on {', '.join(METRICS)}")


Judge register fidelity Phi by condition  (split=val, n=1323 segments, resamples=10000, seed=42)
judge: claude-haiku-4-5  [tag (none)]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition         class  n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
---------------------------------------------------------------------------------------------------------
1     gpt56_sparse_knn  study  1323  3.6251  [3.5843, 3.6659]  0.747  1.000         1 (1.000)   1.00     
2     sparse_knn        study  1323  2.8125  [2.7649, 2.8625]  0.915  1.000         2 (1.000)   2.00     

Score distribution over the rubric (share of parsed segments)
condition         coverage  =1     =2     =3     =4     =5   
-------------------------------------------------------------
gpt56_sparse_knn  1.0000    0.015  0.076  0.219  0.649  0.041
sparse_knn        1.0000    0.067  0.320  0.357  0.246  0.010

Adjacent ranks, paired bootstrap on the shared

---
## 12 — The paired bootstrap

In [57]:
BOOT_PATHS = {}
for metric in METRICS:
    path = f'results/bootstrap_{metric}_{ARM_NAME}_{SPLIT}.json'
    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric, '--conditions', *READOUT,
                        '--split', SPLIT, '--pairs', f'{ARM_NAME}:{REF}',
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--out', path], check=False)
    assert r.returncode == 0, f'{metric} bootstrap exited {r.returncode}'
    BOOT_PATHS[metric] = path

wrote results/bootstrap_chrf_gpt56_sparse_knn_val.json

chrf paired bootstrap  (resamples=10000, split=val)
comparison                     n     diff     ci95                p    sig
--------------------------------------------------------------------------
sparse_knn - gpt56_sparse_knn  1323  -11.097  [-11.760, -10.437]  0.0  *  
gpt56_sparse_knn - sparse_knn  1323  11.097   [10.437, 11.760]    0.0  *  

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_bleu_gpt56_sparse_knn_val.json

bleu paired bootstrap  (resamples=10000, split=val)
comparison                     n     diff     ci95               p    sig
-------------------------------------------------------------------------
sparse_knn - gpt56_sparse_knn  1323  -10.374  [-11.153, -9.632]  0.0  *  
gpt56_sparse_knn - sparse_knn  1323  10.374   [9.632, 11.153]    0.0  *  

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_comet_gpt56_sparse_knn_val.json

comet paired bo

---
## 13 — Read-out by dose

In [58]:
from src.eval.bootstrap import _load_segment_scores, paired_bootstrap

STRATA = {
    f'full dose (n_sparse = {SLOTS})': [i for i, t in enumerate(TRACES) if t['n_sparse'] == SLOTS],
    'all routed': [i for i, t in enumerate(TRACES) if t['route'] != 'dense'],
    'all segments': list(range(len(VAL))),
}
FLOOR = {'judge': 0.058, 'comet': 0.005}
print({k: len(v) for k, v in STRATA.items()})

{'full dose (n_sparse = 4)': 563, 'all routed': 1202, 'all segments': 1323}


In [59]:
SCORES = {}
for metric in METRICS:
    scores, sources = _load_segment_scores(metric, READOUT, OUT, SPLIT, None)
    for cond in READOUT:
        assert len(scores[cond]) == len(VAL), (metric, cond, len(scores[cond]))
        if sources.get(cond) is not None:
            assert sources[cond] == SRC, f'{metric}/{cond}: segment order is not the eval order'
    SCORES[metric] = scores
print('per-segment scores aligned to the eval order for', ', '.join(SCORES))

per-segment scores aligned to the eval order for chrf, bleu, comet, judge


In [60]:
PLACES = {'chrf': 2, 'bleu': 2, 'comet': 4, 'judge': 4}

for metric in METRICS:
    p = PLACES[metric]
    print(f'\n{metric}   {ARM_NAME} - {REF}')
    for label, idx in STRATA.items():
        d = paired_bootstrap([SCORES[metric][ARM_NAME][i] for i in idx],
                             [SCORES[metric][REF][i] for i in idx],
                             n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)
        line = (f"  {label:26s} n={d['n']:5d}  {d['diff']:+.{p}f} "
                f"[{d['ci_low']:+.{p}f}, {d['ci_high']:+.{p}f}]  p={d['p_value']:.4f} "
                f"{'*' if d['significant'] else ' '}")
        if metric in FLOOR:
            f = FLOOR[metric] * (len(VAL) / d['n']) ** 0.5
            line += f"  floor {f:.{p}f}{'' if abs(d['diff']) >= f else '  (under)'}"
        print(line)


chrf   gpt56_sparse_knn - sparse_knn
  full dose (n_sparse = 4)   n=  563  +11.11 [+10.30, +11.93]  p=0.0000 *
  all routed                 n= 1202  +11.05 [+10.38, +11.72]  p=0.0000 *
  all segments               n= 1323  +11.10 [+10.44, +11.76]  p=0.0000 *

bleu   gpt56_sparse_knn - sparse_knn
  full dose (n_sparse = 4)   n=  563  +11.53 [+10.45, +12.63]  p=0.0000 *
  all routed                 n= 1202  +10.44 [+9.67, +11.25]  p=0.0000 *
  all segments               n= 1323  +10.37 [+9.63, +11.15]  p=0.0000 *

comet   gpt56_sparse_knn - sparse_knn
  full dose (n_sparse = 4)   n=  563  +0.0550 [+0.0507, +0.0594]  p=0.0000 *  floor 0.0077
  all routed                 n= 1202  +0.0617 [+0.0577, +0.0656]  p=0.0000 *  floor 0.0052
  all segments               n= 1323  +0.0634 [+0.0593, +0.0674]  p=0.0000 *  floor 0.0050

judge   gpt56_sparse_knn - sparse_knn
  full dose (n_sparse = 4)   n=  563  +0.8401 [+0.7655, +0.9147]  p=0.0000 *  floor 0.0889
  all routed                 n= 1202  +0